# Dò tham số sinh văn bản — tuần 5

Notebook này **không huấn luyện gì cả**. Nó nạp checkpoint `train_20k` đã có, rồi sinh
lại bản tóm tắt trên tập `tune` với vài bộ tham số sinh khác nhau, và chấm điểm từng bộ.

## Vì sao dò trên `tune` chứ không trên `val`

`val` là tập theo dõi trong lúc huấn luyện — mọi quyết định về mô hình đã nhìn nó rồi.
Chọn tham số sinh trên đúng tập ấy là để chính khâu chọn học thuộc `val`, và con số
báo cáo sau đó sẽ đẹp hơn thực tế. `tune` (500 bài) được đóng băng từ tuần 2 cho việc
này, rời hẳn `val` và `test`.

## Vì sao tách khỏi notebook huấn luyện

Kaggle chạy **toàn bộ** ô khi Save & Run All. Nếu thêm khâu dò tham số vào notebook
huấn luyện thì mỗi lần dò sẽ huấn luyện lại `train_20k` — bốn giờ GPU cho một việc chỉ
cần vài phút.

## Trước khi chạy: Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |

Checkpoint đến từ output của notebook `dl-summarisevn-vit5` (khai trong
`kernel-metadata.json`, mục `kernel_sources`). Kaggle gắn nó vào `/kaggle/input/`.

In [ ]:
# ==== CHI SUA O NAY ===================================================
EVAL_SPLIT = "tune"          # KHONG doi thanh val hay test
NAME = "vit5-base-train_20k" # ten he thong trong bang ket qua
CKPT_GLOB = "/kaggle/input/**/VietAI_vit5-base_train_20k/final"

# Luoi tham so. Ban tom tat hien dai 30-31 am tiet trong khi sapo that dai 35,
# tuc dang HUT recall -> dò về phía sinh dài hơn.
#   length_penalty > 1 : beam search uu tien chuoi dai hon
#   min_length        : chan cung do dai toi thieu (token)
GRID = [
    {"length_penalty": 1.0, "min_length": 0},    # mac dinh, de lam moc
    {"length_penalty": 1.5, "min_length": 0},
    {"length_penalty": 2.0, "min_length": 0},
    {"length_penalty": 1.0, "min_length": 20},
    {"length_penalty": 1.5, "min_length": 20},
    {"length_penalty": 2.0, "min_length": 20},
]
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"
OUT = "/kaggle/working/sweep"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print(f"{len(GRID)} cau hinh tren tap {EVAL_SPLIT}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Khong ra duoc Internet. Settings > Internet > On.") from e
print("Moi truong: OK")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1
print("cwd:", os.getcwd())

In [ ]:
# Tim checkpoint trong /kaggle/input. Bao loi RO RANG neu khong thay: nguyen nhan gan
# nhu luon la quen gan output cua notebook huan luyen lam input.
import glob

found = sorted(glob.glob(CKPT_GLOB, recursive=True))
if not found:
    co_gi = sorted(glob.glob("/kaggle/input/*/*"))[:20]
    raise RuntimeError(
        f"Khong thay checkpoint khop {CKPT_GLOB}.\n"
        "Vao Add-ons > Add data > Your Work, them output cua notebook "
        "dl-summarisevn-vit5 (ban chay train_20k).\n"
        f"Hien /kaggle/input co: {co_gi}"
    )
CKPT = found[0]
print("Checkpoint:", CKPT)
!ls -la {CKPT} | head -8

## Chạy lưới tham số

Mỗi cấu hình là một lần gọi `vit5.py --no-train`: nạp checkpoint, sinh 500 bản tóm tắt,
chấm điểm, ghi ra `results/`. Tên file mang theo `lp` và `min` nên các cấu hình không
đè lên nhau; cấu hình mặc định (`lp=1.0, min=0`) không có hậu tố nào.

Không dùng `--eval-limit`: chấm thiếu bài thì các cấu hình không so cặp được với nhau.

In [ ]:
import time

t0 = time.time()
for i, g in enumerate(GRID, 1):
    cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --no-train "
           f"--model {CKPT} --name {NAME} --eval-split {EVAL_SPLIT} "
           f"--length-penalty {g['length_penalty']} --min-length {g['min_length']} "
           f"--out {OUT}")
    print(f"\n===== [{i}/{len(GRID)}] {g} =====")
    print(cmd)
    !{cmd}
    check(_exit_code, f"cau hinh {i} {g}")
print(f"\nXong {len(GRID)} cau hinh trong {(time.time() - t0) / 60:.1f} phut.")

In [ ]:
# Bang so sanh cac cau hinh. `table()` la dung ham cham diem cua du an, khong tu tinh lai.
import json, pathlib, sys
sys.path.insert(0, "src")
from eval.report import table

rows = []
for p in sorted(pathlib.Path("results/tables").glob(f"{NAME}_{EVAL_SPLIT}_in1024*.json")):
    if p.name.endswith("_run.json"):
        continue
    r = json.loads(p.read_text(encoding="utf-8"))[0]
    # Nhan hau to cua ten file lam ten hang: do la cau hinh sinh ra no.
    hau_to = p.stem.replace(f"{NAME}_{EVAL_SPLIT}_in1024", "").strip("_") or "mac dinh"
    rows.append({**r, "name": hau_to})
print(table(rows))
print("\nSapo that dai trung binh 35 am tiet — cot 'Do dai' cang gan 35 cang tot,")
print("nhung tieu chi chon van la rouge1/rouge2, do dai chi de giai thich.")

In [ ]:
# Gom ket qua (khong gom trong so) thanh mot zip de tai ve tu tab Output.
import zipfile

picked = sorted(p for p in pathlib.Path("results").rglob("*.json") if p.name.startswith(f"{NAME}_{EVAL_SPLIT}_"))
if not picked:
    raise RuntimeError("Khong thay file ket qua nao cua khao sat tham so.")
zpath = f"/kaggle/working/ket_qua_sweep_{NAME}_{EVAL_SPLIT}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():86s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)

## Sau khi chạy

1. Tải `ket_qua_sweep_*.zip` ở tab **Output**, giải nén tại thư mục gốc repo.
2. So các cấu hình bằng bootstrap ghép cặp (`eval.report.compare`) — chênh lệch nhỏ
   giữa hai cấu hình rất dễ là nhiễu, và cả sáu đều chấm trên đúng 500 bài của `tune`
   nên ghép cặp hợp lệ.
3. Cấu hình thắng mới được đem sang `val`/`test`. **Đừng** chọn cấu hình dựa trên `val`.